# HyDE — Hypothetical Document Embeddings
### Embed a fabricated answer instead of the raw question

Corpus: `OWASP Top 10 for LLM Applications (2025)` — 10 named risk categories (LLM01–LLM10) sharing vocabulary like “risk”, “attack”, “model”, which is exactly what makes naive retrieval struggle.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

C:\Users\shiva\AppData\Local\Temp\ipykernel_19156\1805325906.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

incorrect startxref pointer(1)


parsing for Object Streams


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Baseline — embed the raw question
A short question and a long technical passage occupy different regions of embedding space, even when they're about the same topic.

In [4]:
query = "Can someone mess with the data my model learns from?"

print("Baseline — embedding the raw question:")
for doc in vector_store.similarity_search(query, k=3):
    print(f"page {doc.metadata['page']}: {doc.page_content[:150]}...")

Baseline — embedding the raw question:
page 19: OWASP Top 10 for LLM Applications v2.0
16genai.owasp.org
LLM04: Data and Model Poisoning
Description
Data poisoning occurs when pre-training, fine-tun...
page 20: Example Attack Scenarios
Scenario #1
An attacker biases the model's outputs by manipulating training data or using prompt
injection techniques, spread...
page 10: 2. Proprietary Algorithm Exposure
Poorly configured model outputs can reveal proprietary algorithms or data. Revealing training
data can expose models...


## Step 3: Generate a hypothetical answer
Ask the LLM to write a plausible (possibly wrong) passage that *sounds like* it came from the source document. We never check its factual accuracy — only its embedding is used.

In [5]:
hyde_prompt = f"""Write a short, technical passage (3-4 sentences) that plausibly answers this question,
as if it were an excerpt from a security best-practices document. Do not mention that this is hypothetical.

Question: {query}
Passage:"""

hypothetical_doc = llm.invoke(hyde_prompt).content.strip()
print(hypothetical_doc)

When utilizing sentence transformer models, it is essential to consider the potential for adversarial manipulation of training data. An attacker could potentially inject malicious or biased text into the dataset, which would be propagated through the model's learned representations and impact its performance on downstream tasks. To mitigate this risk, it is recommended that organizations implement robust data validation and verification procedures, such as data profiling and anomaly detection, to identify and remove suspicious or tampered content from the training set. Additionally, using techniques like data augmentation and regularization can help to reduce the model's reliance on specific, vulnerable examples.


## Step 4: Embed the hypothetical answer and retrieve

In [6]:
print("HyDE — embedding the hypothetical passage:")
for doc in vector_store.similarity_search(hypothetical_doc, k=3):
    print(f"page {doc.metadata['page']}: {doc.page_content[:150]}...")

HyDE — embedding the hypothetical passage:


page 20: Example Attack Scenarios
Scenario #1
An attacker biases the model's outputs by manipulating training data or using prompt
injection techniques, spread...
page 10: 2. Proprietary Algorithm Exposure
Poorly configured model outputs can reveal proprietary algorithms or data. Revealing training
data can expose models...
page 19: originate. Data poisoning is considered an integrity attack since tampering with training data
impacts the model's ability to make accurate prediction...


## Step 5: Generate the final answer
Retrieve using the hypothetical doc, but still answer the user's real question with the *real* retrieved chunks.

In [7]:
retrieved_docs = vector_store.similarity_search(hypothetical_doc, k=3)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

print(llm.invoke(prompt).content)

Yes, someone can mess with the data your model learns from. This is known as "data poisoning" and it involves introducing malicious or biased data into the training set, which can lead to biased or inaccurate outputs from the model.


## Try it yourself
1. Compare HyDE against the baseline for a very short query like `"model theft?"`.
2. Make the hypothetical-answer prompt longer/more detailed and see if retrieval improves further.
3. Try HyDE on a question the document doesn't actually answer — what happens?